Predictive Maintenance using Machine Learning for Bearing RUL Prediction

This notebook develops a predictive maintenance framework for bearing condition monitoring using vibration-based Remaining Useful Life (RUL) prediction and health-state classification.

Multiple machine learning regression models are evaluated using leakage-safe validation, and the best model is selected for prediction on unseen test bearings.

Import Required Libraries

This section imports the required Python libraries for:

- data handling
- feature engineering
- machine learning
- evaluation
- visualization
- model saving

In [37]:
import os
import pandas as pd
import numpy as np
from scipy.stats import kurtosis, skew
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, ExtraTreesRegressor
from sklearn.svm import SVR
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import joblib
import warnings
warnings.filterwarnings("ignore")
print("Libraries loaded!")

Libraries loaded!


Load Training Feature Dataset

The training dataset contains extracted vibration-based statistical features and corresponding Remaining Useful Life (RUL) labels for each bearing sample.

In [38]:
df = pd.read_csv("train_features.csv")
print(f"Loaded: {df.shape[0]} rows, {df.shape[1]} columns")
print(f"Bearings: {df['bearing'].unique()}")
print(df.head(3))

Loaded: 7534 rows, 16 columns
Bearings: <StringArray>
['Bearing1_1', 'Bearing1_2', 'Bearing2_1', 'Bearing2_2', 'Bearing3_1',
 'Bearing3_2']
Length: 6, dtype: str
      rms_x  peak_x  kurtosis_x    skew_x     std_x   crest_x     rms_y  \
0  0.561746   2.010   -0.131465 -0.004711  0.561735  3.578132  0.435801   
1  0.535112   1.915   -0.084646 -0.025791  0.535083  3.578687  0.420968   
2  0.531158   1.901    0.033388 -0.005016  0.531142  3.578971  0.425605   

   peak_y  kurtosis_y    skew_y     std_y   crest_y  time_step  total_steps  \
0   1.591   -0.035080  0.002713  0.435797  3.650745          0         2803   
1   1.666    0.150620  0.077380  0.420967  3.957542          1         2803   
2   1.584   -0.061552 -0.024393  0.425600  3.721758          2         2803   

    RUL     bearing  
0  2802  Bearing1_1  
1  2801  Bearing1_1  
2  2800  Bearing1_1  


Inspect Dataset Structure

This step examines the dataset dimensions, feature columns, and sample records to understand the available information for model training.

Normalize Remaining Useful Life

The Remaining Useful Life (RUL) is normalized relative to the total bearing lifetime:

RUL_norm = RUL / total_steps

This allows the model to learn relative degradation progression instead of absolute lifetime values, improving generalization across different bearings.

In [39]:
df["RUL_norm"]      = df["RUL"] / df["total_steps"]
df["life_fraction"] = df["time_step"] / df["total_steps"]

print("RUL normalized to [0, 1]")
print(df[["bearing", "RUL", "total_steps", "RUL_norm"]].head())

RUL normalized to [0, 1]
      bearing   RUL  total_steps  RUL_norm
0  Bearing1_1  2802         2803  0.999643
1  Bearing1_1  2801         2803  0.999286
2  Bearing1_1  2800         2803  0.998930
3  Bearing1_1  2799         2803  0.998573
4  Bearing1_1  2798         2803  0.998216


Prepare Features and Target Variables

The vibration-based statistical features are selected as input variables for machine learning.

The target variable is the normalized Remaining Useful Life (RUL), which improves generalization across bearings with different lifetimes.

The bearing identifier is used for leakage-safe grouped validation.

In [40]:
feature_cols = [
    "rms_x", "peak_x", "kurtosis_x", "skew_x", "std_x", "crest_x",
    "rms_y", "peak_y", "kurtosis_y", "skew_y", "std_y", "crest_y"
]

X      = df[feature_cols].values
y      = df["RUL_norm"].values
groups = df["bearing"].values

print(f"Features: {len(feature_cols)}")
print(f"Samples:  {len(X)}")
print(f"Feature list: {feature_cols}")

Features: 12
Samples:  7534
Feature list: ['rms_x', 'peak_x', 'kurtosis_x', 'skew_x', 'std_x', 'crest_x', 'rms_y', 'peak_y', 'kurtosis_y', 'skew_y', 'std_y', 'crest_y']


Leakage-Safe Train-Test Split

The dataset is split using GroupShuffleSplit based on bearing IDs.

This prevents samples from the same bearing from appearing in both training and validation sets, avoiding data leakage and ensuring realistic predictive maintenance evaluation.

In [41]:
gss = GroupShuffleSplit(n_splits=1, test_size=0.33, random_state=42)
train_idx, val_idx = next(gss.split(X, y, groups))

X_train, X_val = X[train_idx], X[val_idx]
y_train, y_val = y[train_idx], y[val_idx]

print(f"Train bearings : {set(groups[train_idx])}")
print(f"Val bearings   : {set(groups[val_idx])}")
print(f"Overlap        : {set(groups[train_idx]) & set(groups[val_idx])} ← must be empty")
print(f"Train samples  : {len(X_train)}")
print(f"Val samples    : {len(X_val)}")

Train bearings : {'Bearing2_2', 'Bearing3_1', 'Bearing2_1', 'Bearing3_2'}
Val bearings   : {'Bearing1_1', 'Bearing1_2'}
Overlap        : set() ← must be empty
Train samples  : 3860
Val samples    : 3674


Train and Compare Multiple Regression Models

Several regression models are evaluated for Remaining Useful Life prediction:

- Random Forest Regressor
- Extra Trees Regressor
- Gradient Boosting Regressor
- Support Vector Regression (SVR)

Each model is evaluated using:

- R² score
- RMSE
- MAE

The best-performing model is selected automatically based on validation R² score.

In [42]:
model_results = {}

# 1. Random Forest
print("Training Random Forest...")
rf = RandomForestRegressor(
    n_estimators=300, max_depth=8,
    min_samples_leaf=10, random_state=42, n_jobs=-1
)
rf.fit(X_train, y_train)
rf_r2   = r2_score(y_val, rf.predict(X_val))
rf_rmse = np.sqrt(mean_squared_error(y_val, rf.predict(X_val)))
rf_mae  = mean_absolute_error(y_val, rf.predict(X_val))
model_results["Random Forest"] = {"model": rf, "r2": rf_r2, "rmse": rf_rmse, "mae": rf_mae}
print(f"  R²: {rf_r2:.4f} | RMSE: {rf_rmse:.4f} | MAE: {rf_mae:.4f}")

# 2. Extra Trees
print("Training Extra Trees...")
et = ExtraTreesRegressor(
    n_estimators=300, max_depth=8,
    min_samples_leaf=10, random_state=42, n_jobs=-1
)
et.fit(X_train, y_train)
et_r2   = r2_score(y_val, et.predict(X_val))
et_rmse = np.sqrt(mean_squared_error(y_val, et.predict(X_val)))
et_mae  = mean_absolute_error(y_val, et.predict(X_val))
model_results["Extra Trees"] = {"model": et, "r2": et_r2, "rmse": et_rmse, "mae": et_mae}
print(f"  R²: {et_r2:.4f} | RMSE: {et_rmse:.4f} | MAE: {et_mae:.4f}")

# 3. Gradient Boosting
print("Training Gradient Boosting...")
gb = GradientBoostingRegressor(
    n_estimators=300, learning_rate=0.05,
    max_depth=4, random_state=42
)
gb.fit(X_train, y_train)
gb_r2   = r2_score(y_val, gb.predict(X_val))
gb_rmse = np.sqrt(mean_squared_error(y_val, gb.predict(X_val)))
gb_mae  = mean_absolute_error(y_val, gb.predict(X_val))
model_results["Gradient Boosting"] = {"model": gb, "r2": gb_r2, "rmse": gb_rmse, "mae": gb_mae}
print(f"  R²: {gb_r2:.4f} | RMSE: {gb_rmse:.4f} | MAE: {gb_mae:.4f}")

# 4. SVR
print("Training SVR...")
svr_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("svr", SVR(kernel="rbf", C=10, epsilon=0.1))
])
svr_pipe.fit(X_train, y_train)
svr_r2   = r2_score(y_val, svr_pipe.predict(X_val))
svr_rmse = np.sqrt(mean_squared_error(y_val, svr_pipe.predict(X_val)))
svr_mae  = mean_absolute_error(y_val, svr_pipe.predict(X_val))
model_results["SVR"] = {"model": svr_pipe, "r2": svr_r2, "rmse": svr_rmse, "mae": svr_mae}
print(f"  R²: {svr_r2:.4f} | RMSE: {svr_rmse:.4f} | MAE: {svr_mae:.4f}")

# Summary table
print("\n" + "="*55)
print(f"{'Model':<20} {'R²':>8} {'RMSE':>8} {'MAE':>8}")
print("="*55)
for name, res in model_results.items():
    print(f"{name:<20} {res['r2']:>8.4f} {res['rmse']:>8.4f} {res['mae']:>8.4f}")
print("="*55)

Training Random Forest...
  R²: -0.3861 | RMSE: 0.3399 | MAE: 0.2767
Training Extra Trees...
  R²: 0.3221 | RMSE: 0.2377 | MAE: 0.1941
Training Gradient Boosting...
  R²: -0.3477 | RMSE: 0.3351 | MAE: 0.2637
Training SVR...
  R²: -8.8975 | RMSE: 0.9082 | MAE: 0.5537

Model                      R²     RMSE      MAE
Random Forest         -0.3861   0.3399   0.2767
Extra Trees            0.3221   0.2377   0.1941
Gradient Boosting     -0.3477   0.3351   0.2637
SVR                   -8.8975   0.9082   0.5537


Select Best Performing Model:

The regression model with the highest validation R² score is selected as the final predictive maintenance model.

This automated selection ensures the most suitable model is used for deployment on unseen bearings.

Retrain Final Model on All Training Bearings:

After validation, the selected best model is retrained using all available training bearings.

This final model is then used for prediction on unseen bearings from the Test_set dataset.

Save Final Trained Model:

The final trained predictive maintenance model is saved as a serialized `.pkl` file for future reuse and deployment.

In [43]:
best_name  = max(model_results, key=lambda k: model_results[k]["r2"])
best_model = model_results[best_name]["model"]
best_r2    = model_results[best_name]["r2"]

print(f"✅ Best model: {best_name} (R²={best_r2:.4f})")

# Retrain on ALL training data
best_model.fit(X, y)
joblib.dump(best_model, "Final_best_rul_model.pkl")
joblib.dump(feature_cols, "Final_feature_cols_v2.pkl")
print("Saved: Final_best_rul_model.pkl")

✅ Best model: Extra Trees (R²=0.3221)
Saved: Final_best_rul_model.pkl


Load Unseen Test Bearings:

- The unseen Test_set bearings are loaded and processed using the same vibration feature extraction pipeline used during training.
- These bearings were not used during model validation or training.

Extract Statistical Vibration Features:

Statistical vibration features are extracted from the raw accelerometer signals in both x and y directions.

Extracted features include:

- RMS
- Peak value
- Kurtosis
- Skewness
- Standard deviation
- Crest factor

These features characterize bearing degradation behavior for predictive maintenance analysis.

In [44]:
def extract_features(filepath):
    df_raw = pd.read_csv(filepath, header=None)
    ax = df_raw[4].values
    ay = df_raw[5].values if 5 in df_raw.columns else ax
    features = {}
    for name, sig in [("x", ax), ("y", ay)]:
        features[f"rms_{name}"]      = np.sqrt(np.mean(sig**2))
        features[f"peak_{name}"]     = np.max(np.abs(sig))
        features[f"kurtosis_{name}"] = kurtosis(sig)
        features[f"skew_{name}"]     = skew(sig)
        features[f"std_{name}"]      = np.std(sig)
        features[f"crest_{name}"]    = np.max(np.abs(sig)) / (np.sqrt(np.mean(sig**2)) + 1e-10)
    return features

test_path = "Test_set"
all_test  = []
for bearing in sorted(os.listdir(test_path)):
    path  = os.path.join(test_path, bearing)
    files = sorted([f for f in os.listdir(path) if f.startswith("acc_")])
    records = []
    for i, f in enumerate(files):
        feats = extract_features(os.path.join(path, f))
        feats["time_step"]   = i
        feats["total_steps"] = len(files)
        feats["bearing"]     = bearing
        records.append(feats)
    all_test.append(pd.DataFrame(records))
    print(f"  {bearing}: {len(files)} files")

test_df = pd.concat(all_test, ignore_index=True)
print(f"\nTotal test rows: {test_df.shape[0]}")

  Bearing1_3: 1802 files
  Bearing1_4: 1139 files
  Bearing1_5: 2302 files
  Bearing1_6: 2302 files
  Bearing1_7: 1502 files
  Bearing2_3: 1202 files
  Bearing2_4: 612 files
  Bearing2_5: 2002 files
  Bearing2_6: 572 files
  Bearing2_7: 172 files
  Bearing3_3: 352 files

Total test rows: 13959


Predict Remaining Useful Life on Unseen Bearings:

The final trained model predicts the normalized Remaining Useful Life (RUL) for unseen bearings from the Test_set dataset.

Prediction uncertainty and confidence bounds are also estimated for maintenance interpretation.

In [45]:
X_test = test_df[feature_cols].values

# Per-bearing uncertainty
if best_name in ["Random Forest", "Extra Trees"]:
    preds_trees = np.array([tree.predict(X_test) for tree in best_model.estimators_])
    y_pred_norm = preds_trees.mean(axis=0)
    y_std_norm  = preds_trees.std(axis=0)
else:
    y_pred_norm = best_model.predict(X_test)
    y_std_norm  = np.zeros(len(y_pred_norm))
    for bearing in test_df["bearing"].unique():
        mask   = test_df["bearing"] == bearing
        preds  = y_pred_norm[mask]
        y_std_norm[mask] = pd.Series(preds).rolling(10, min_periods=1).std().fillna(0).values

# Smooth predictions with rolling average
smoothed_pred = []
smoothed_std  = []
for bearing in test_df["bearing"].unique():
    mask  = test_df["bearing"] == bearing
    preds = pd.Series(y_pred_norm[mask]).rolling(20, min_periods=1).mean().values
    stds  = pd.Series(y_std_norm[mask]).rolling(20, min_periods=1).mean().values
    smoothed_pred.extend(preds)
    smoothed_std.extend(stds)

test_df["pred_norm"] = np.clip(smoothed_pred, 0, 1)
test_df["std_norm"]  = smoothed_std

# Convert to seconds
bearing_total = test_df.groupby("bearing")["total_steps"].first()
test_df["Predicted_RUL_s"] = test_df["pred_norm"] * test_df["bearing"].map(bearing_total) * 10
test_df["Uncertainty_s"]   = test_df["std_norm"]  * test_df["bearing"].map(bearing_total) * 10
test_df["Lower_RUL_s"]     = np.maximum(test_df["Predicted_RUL_s"] - test_df["Uncertainty_s"], 0)
test_df["Upper_RUL_s"]     = test_df["Predicted_RUL_s"] + test_df["Uncertainty_s"]

def classify_health(rul_norm):
    if rul_norm <= 0.20:
        return "Imminent failure"
    elif rul_norm <= 0.50:
        return "Wear detectable"
    else:
        return "Non-critical"

test_df["Health_State"] = test_df["pred_norm"].apply(classify_health)
print("Predictions done!")

Predictions done!


Health-State Classification:

The predicted Remaining Useful Life values are converted into maintenance-oriented health states:

- Non-critical
- Wear detectable
- Imminent failure

This converts continuous RUL predictions into interpretable maintenance decisions for predictive maintenance applications.

In [46]:
actual_rul = {
    "Bearing1_3": 5730, "Bearing1_4": 339,  "Bearing1_5": 1610,
    "Bearing1_6": 1460, "Bearing1_7": 7570, "Bearing2_3": 7530,
    "Bearing2_4": 1390, "Bearing2_5": 3090, "Bearing2_6": 1290,
    "Bearing2_7": 580,  "Bearing3_3": 820
}

latest = test_df.sort_values("time_step").groupby("bearing").last().reset_index()

print(f"Best model: {best_name}\n")
print(f"{'Bearing':<12} {'Predicted(s)':>13} {'Actual(s)':>10} {'Error%':>8} {'Uncertainty':>13} {'Health'}")
print("-" * 85)

errors = []
for _, row in latest.iterrows():
    b      = row["bearing"]
    pred_s = row["Predicted_RUL_s"]
    act_s  = actual_rul[b]
    unc_s  = row["Uncertainty_s"]
    health = row["Health_State"]
    err    = abs(pred_s - act_s) / act_s * 100
    errors.append(err)
    print(f"{b:<12} {pred_s:>13.0f} {act_s:>10} {err:>7.1f}% {unc_s:>12.0f}s  {health}")

print("-" * 85)
print(f"Average Error: {np.mean(errors):.1f}%")
print(f"Best  Error:   {np.min(errors):.1f}%")
print(f"Worst Error:   {np.max(errors):.1f}%")

Best model: Extra Trees

Bearing       Predicted(s)  Actual(s)   Error%   Uncertainty Health
-------------------------------------------------------------------------------------
Bearing1_3            3342       5730    41.7%         2304s  Imminent failure
Bearing1_4              88        339    74.1%          131s  Imminent failure
Bearing1_5           10225       1610   535.1%         4267s  Wear detectable
Bearing1_6           11596       1460   694.2%         3022s  Non-critical
Bearing1_7            8903       7570    17.6%         1959s  Non-critical
Bearing2_3            6079       7530    19.3%         1601s  Non-critical
Bearing2_4            3635       1390   161.5%          684s  Non-critical
Bearing2_5            9987       3090   223.2%         3260s  Wear detectable
Bearing2_6            1782       1290    38.1%         1181s  Wear detectable
Bearing2_7            1152        580    98.6%          132s  Non-critical
Bearing3_3            2307        820   181.3%        

Save Predictive Maintenance Results:

The final prediction results for unseen bearings are saved as a CSV file containing:

- predicted RUL
- uncertainty estimates
- confidence bounds
- health-state classifications

These outputs support condition-based maintenance planning and failure prevention.

In [47]:
test_df.to_csv("Final_best_test_predictions_timeseries.csv", index=False)
latest.to_csv("Final_best_test_predictions_final.csv", index=False)
print("Saved: Final_best_test_predictions_timeseries.csv")
print("Saved: Final_best_test_predictions_final.csv")
print(f"\n Modelling complete! Best model: {best_name}")

Saved: Final_best_test_predictions_timeseries.csv
Saved: Final_best_test_predictions_final.csv

 Modelling complete! Best model: Extra Trees


Final Summary

A complete predictive maintenance framework was developed for bearing condition monitoring using machine learning-based Remaining Useful Life prediction and health-state classification.

The final system:

- avoids data leakage using GroupShuffleSplit
- evaluates multiple regression models
- selects the best-performing model automatically
- predicts on real unseen Test_set bearings
- generates interpretable maintenance-oriented health states

The framework successfully demonstrates a leakage-safe industrial predictive maintenance workflow for bearing prognostics.

Among the evaluated regression models, Extra Trees Regressor achieved the best validation performance and was selected as the final predictive maintenance model.